# autoalgo benchmark notebooks: start here

autoalgo is a speed experiment. Each algorithm is built more than one way and raced against scikit-learn, to answer one question: **is it worth building a faster engine, or is scikit-learn already as fast as this machine allows?**

| Notebook | Algorithm | Bottleneck it tests | autoalgo engines |
|---|---|---|---|
| `01_knn` | k-nearest neighbors | matrix math, then picking the k smallest (memory) | numpy, rust, rust-blas, mlx |
| `02_kmeans` | k-means | matrix math repeated every iteration, plus summing rows per cluster | numpy, rust, mlx |
| `03_ridge` | ridge regression | one big matrix multiply, then a tiny solve | numpy, mlx |
| `04_pca` | PCA | one big matrix multiply, then a tiny eigen-solve, *or* a full SVD depending on the solver | numpy, mlx |
| `05_decision_tree` | decision tree | branchy, integer counting: no matrix math at all | numpy, rust |
| `06_summary` | all of the above | the scoreboard, and what it means for the library | |

**What the engine names mean**

- `autoalgo-numpy`: autoalgo built *on top of* existing libraries. Python plus NumPy, which hands matrix math to Apple Accelerate.
- `autoalgo-rust`: compiled Rust (`src/`), every core, with the GIL released. For k-means it also calls Accelerate for the matrix math.
- `autoalgo-rust-blas` (KNN only): Rust that hands the distance math to Accelerate, which can use the M4's matrix units.
- `autoalgo-mlx`: pure Python on the Apple GPU through MLX.

Some algorithms have no Rust or MLX engine on purpose. Each notebook says why.

## How to run these yourself

1. Open the `autoalgo` folder in your IDE.
2. Pick the interpreter/kernel at `autoalgo/.venv/bin/python` (everything is already installed there).
3. If you change anything in `src/` (the Rust code), rebuild first from the `autoalgo` folder: `.venv/bin/maturin develop --release`
4. Run the notebooks top to bottom. Each one saves its numbers to `notebooks/results/`, which `06_summary` reads.

Run times on the M4: about 1 to 4 minutes per notebook. Every notebook has a `QUICK = False` switch near the top; set it to `True` for a fast smoke run on smaller data.

In [1]:
import sys
sys.path.insert(0, ".")  # so bench_utils (next to this notebook) imports

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import autoalgo as aa
from bench_utils import (machine_info, timeit, load_covtype, add_speedup,
                         save_results, line_chart, fmt_ms)

pd.set_option("display.float_format", "{:,.4f}".format)
%matplotlib inline
pd.Series(machine_info())

chip                                  Apple M4
cores             4 performance + 6 efficiency
ram_gb                                      24
os         macOS-26.6.1-arm64-arm-64bit-Mach-O
python                                  3.13.5
numpy                       2.5.4 (accelerate)
sklearn                                  1.9.1
mlx                                     0.32.4
dtype: object

## Sanity check: every engine imports and agrees on a tiny problem

In [2]:
from sklearn.datasets import load_iris
from sklearn.neighbors import KNeighborsClassifier as SkKNN

X, y = load_iris(return_X_y=True)
ref = SkKNN(5).fit(X, y).predict(X)
pd.DataFrame({
    backend: {"matches scikit-learn": bool((aa.KNeighborsClassifier(5, backend=backend).fit(X, y).predict(X) == ref).all())}
    for backend in ["numpy", "rust", "rust-blas", "mlx"]
}).T

,matches scikit-learn
numpy,True
rust,True
rust-blas,True
mlx,True
